# 10 — One-shot LIBERO-PRO verifier V2 confirmation

Run only after notebook 09 registers the final bundle and all 160 confirmatory
groups are complete. This opens the sealed outcomes once and applies the
predeclared registration gates.

## 1. Setup

In [ ]:
import os, subprocess, sys
try:
    from google.colab import userdata
    for key in ("SUPABASE_URL", "SUPABASE_SERVICE_KEY", "HF_TOKEN", "WANDB_API_KEY"):
        value = userdata.get(key)
        if value: os.environ[key] = value
    repo_dir = "/content/cs159-sp26"
    gh_pat = userdata.get("GH_PAT")
    repo_url = f"https://{gh_pat}@github.com/ArjunS07/cs159-sp26.git"
    if not os.path.isdir(os.path.join(repo_dir, ".git")):
        subprocess.run(["git", "clone", "--branch", "main", repo_url, repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only", "origin", "main"], check=True)
except ImportError:
    repo_dir = os.path.abspath("..") if os.path.basename(os.getcwd()) == "pnp-vla" else os.getcwd()
package_dir = os.path.join(repo_dir, "pnp-vla")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                package_dir + "[sim,analysis]"], check=True)
if package_dir not in sys.path: sys.path.insert(0, package_dir)
import pnp
print("Loaded pnp from:", pnp.__file__)

## 2. Load the frozen bundle and sealed candidates

In [ ]:
import json
from pathlib import Path
import torch
from pnp.store import SupabaseStore
from pnp.verifier import *

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT=Path(package_dir)/"analysis_outputs"/"verifier_v2"; OUTPUT.mkdir(parents=True,exist_ok=True)
store=SupabaseStore()
registered=(store.client.table("verifier_models").select("verifier_id,created_at")
            .eq("experiment","state-conditioned-verifier-v2")
            .order("created_at",desc=True).limit(1).execute().data or [])
assert registered,"no state-conditioned-verifier-v2 checkpoint is registered"
verifier_id=registered[0]["verifier_id"]
checkpoint,row=store.load_verifier(verifier_id)
examples=load_candidate_examples(
    store,"verifier-v2-pro-confirmatory",cache_dir=OUTPUT/"confirmatory_cache")
audit=validate_candidate_groups(examples,expected_candidates=12)
assert audit["groups"] >= 150, audit

def load_model(spec,state):
    model=CompactAdvantageVerifier(obs_dim=int(row["obs_dim"]),action_dim=int(row["action_dim"]),
        action_width=64,dropout=spec["dropout"],conditioning=spec["architecture"])
    model.load_state_dict(state); return model.to(DEVICE).eval()

selected_spec=checkpoint["metadata"]["selected_spec"]
selected=load_model(selected_spec,checkpoint["model"])
action=load_model(checkpoint["controls"]["action_only"]["spec"],
                  checkpoint["controls"]["action_only"]["model"])
shuffled=load_model(checkpoint["controls"]["shuffled_actions"]["spec"],
                    checkpoint["controls"]["shuffled_actions"]["model"])
print({"verifier":verifier_id,"integrity":audit})

## 3. Predeclared paired-bootstrap gate

In [ ]:
config=AdvantageTrainConfig(seed=20260728,prefix_length=10)
metrics,selected_records=evaluate_candidate_ranker(
    selected,examples,DEVICE,config=config,return_records=True)
_,action_records=evaluate_candidate_ranker(
    action,examples,DEVICE,config=config,return_records=True)
_,shuffled_records=evaluate_candidate_ranker(
    shuffled,examples,DEVICE,config=config,return_records=True)
action_comparison=paired_candidate_comparison(selected_records,action_records,seed=20260728)
shuffled_comparison=paired_candidate_comparison(
    selected_records,shuffled_records,seed=20260729)
gate=verifier_registration_eligibility(metrics,action_comparison,shuffled_comparison)
report={"verifier_id":verifier_id,"metrics":metrics,
        "vs_action_only":action_comparison,"vs_shuffled_actions":shuffled_comparison,
        "registration_gate":gate}
(OUTPUT/"confirmatory_report.json").write_text(json.dumps(report,indent=2,sort_keys=True))
(store.client.table("verifier_models").update({"metrics_json":report})
 .eq("verifier_id",verifier_id).execute())
print(json.dumps(report,indent=2))
print("REGISTERED" if gate["eligible"] else "EXPLORATORY ONLY — gate not met")